# Entity Recommendations

## Goal

Given an input clinical study, recommend entities that similar studies use and the input study does not.

## "Dumb" vs "Smart" Recommendations

**Dumb recommendations** rank missing entities from co-occurrence in similar studies, then try deviation and network rerankers. They do not use study outcomes.


In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from src.recommendation_engine import experiment_frames
target_study = __import__("src.similarity", fromlist=["protocol_studies"]).protocol_studies()["study_name"].tolist()[0]
print("target", target_study)
frames = experiment_frames(target_study)


target NCT02060721


## Find Top K Similar Studies

In [2]:
display(frames['similar'])

,study_id,similarity
0,NCT02427958,0.664822
1,NCT02879578,0.645054
2,NCT02709655,0.618838
3,NCT02188121,0.616638
4,NCT03461016,0.573085


## Extract Missing Entities

In [3]:
display(frames['missing'].head(15))

,Entity,Frequency,Similar Studies,Avg Similarity
38,INCLUSION_CRITERION- Age requirement,2,"NCT02879578, NCT02709655",0.632
1,ASSESSMENT- Overdose documentation,1,NCT02427958,0.665
0,ASSESSMENT- Efficacy and safety analysis,1,NCT02427958,0.665
3,ENDPOINT- Change in bone mineral density,1,NCT02427958,0.665
4,ENDPOINT- Change in laboratory values,1,NCT02427958,0.665
5,ENDPOINT- Decrease in BA to chronological age ...,1,NCT02427958,0.665
2,ENDPOINT- BMI measurement,1,NCT02427958,0.665
6,ENDPOINT- Improvement in predicted adult height,1,NCT02427958,0.665
7,ENDPOINT- Incidence of polycystic ovarian synd...,1,NCT02427958,0.665
9,ENDPOINT- Suppression of basal estradiol or te...,1,NCT02427958,0.665


## Co-occurrence Pattern Analysis

In [4]:
cooccurrence_df = frames['baseline']
display(cooccurrence_df.head(20))

,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language
0,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,Studies that used INCLUSION_CRITERION- Written...
1,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,Studies that used INCLUSION_CRITERION- Written...
2,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,Studies that used INCLUSION_CRITERION- Written...
3,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,Studies that used INCLUSION_CRITERION- Written...
4,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,Studies that used INCLUSION_CRITERION- Written...
5,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,Studies that used INCLUSION_CRITERION- Written...
6,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,Studies that used INCLUSION_CRITERION- Written...
7,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,Studies that used INCLUSION_CRITERION- Written...
8,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,Studies that used INCLUSION_CRITERION- Written...
9,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,Studies that used INCLUSION_CRITERION- Written...


## Experiment 1: Increased Top K & Entity Deviation Weight

In [5]:
display(frames['experiment_1'].head(20))

,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language
0,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,2,2,2,100.0,"NCT02427958, NCT02752048",INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,2,Studies that used INCLUSION_CRITERION- Written...
1,ENDPOINT- Change in 6MWD,INCLUSION_CRITERION- Age requirement,1,1,3,100.0,NCT02752048,ENDPOINT- Change in 6MWD,INCLUSION_CRITERION- Age requirement,1,Studies that used ENDPOINT- Change in 6MWD als...
2,ENDPOINT- Change in 6MWD,ASSESSMENT- Efficacy and safety analysis,1,1,2,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Efficacy and safety analysis,1,Studies that used ENDPOINT- Change in 6MWD als...
3,ENDPOINT- Change in 6MWD,ASSESSMENT- Safety evaluation,1,1,2,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Safety evaluation,1,Studies that used ENDPOINT- Change in 6MWD als...
4,ENDPOINT- Change in 6MWD,ASSESSMENT- Severity of adverse events,1,1,2,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Severity of adverse events,1,Studies that used ENDPOINT- Change in 6MWD als...
5,ENDPOINT- Change in 6MWD,INTERVENTION- Dose modification,1,1,2,100.0,NCT02752048,ENDPOINT- Change in 6MWD,INTERVENTION- Dose modification,1,Studies that used ENDPOINT- Change in 6MWD als...
6,ENDPOINT- Change in 6MWD,ASSESSMENT- 12-Lead ECG,1,1,1,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- 12-Lead ECG,1,Studies that used ENDPOINT- Change in 6MWD als...
7,ENDPOINT- Change in 6MWD,ASSESSMENT- Assessment of expelled or partial ...,1,1,1,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Assessment of expelled or partial ...,1,Studies that used ENDPOINT- Change in 6MWD als...
8,ENDPOINT- Change in 6MWD,ASSESSMENT- Blood pressure and pulse rate meas...,1,1,1,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Blood pressure and pulse rate meas...,1,Studies that used ENDPOINT- Change in 6MWD als...
9,ENDPOINT- Change in 6MWD,ASSESSMENT- Endocrinological parameters measur...,1,1,1,100.0,NCT02752048,ENDPOINT- Change in 6MWD,ASSESSMENT- Endocrinological parameters measur...,1,Studies that used ENDPOINT- Change in 6MWD als...


## Experiment 2: Specialized Entity Filtering Using Deviation Artifacts

In [6]:
display(frames['experiment_2'].head(20))

,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language,specialized_deviation_score
0,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
1,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
2,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
3,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
4,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Unacceptable risk to health,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Unacceptable risk to health,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
5,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Overdose without symptoms...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Overdose without symptoms...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
6,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Recent investigational co...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,EXCLUSION_CRITERION- Recent investigational co...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
7,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
8,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of peak LH and FSH to pr...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of peak LH and FSH to pr...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0
9,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,Studies that used INCLUSION_CRITERION- Written...,1669.0


## Experiment 3: Co-occurrence Network Analysis

## Network Score

In [7]:
display(frames['experiment_3'].head(20))

,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language,pagerank_suggested,betweenness_suggested,target_overlap,network_score
0,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
1,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
2,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
3,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
4,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
5,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
6,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
7,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
8,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783
9,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783


## Experiment 4: Specialized Network Score (Inverted Global Metrics)

In [8]:
display(frames['experiment_4'].head(20))

,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language,pagerank_suggested,betweenness_suggested,target_overlap,network_score,hub_adjustment,specialized_network_score
0,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
1,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
2,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
3,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
4,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
5,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
6,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
7,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
8,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
9,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
